# Mini Project: Transfer Learning with Keras

Transfer learning is a machine learning technique where a model trained on one task is used as a starting point to solve a different but related task. Instead of training a model from scratch, transfer learning leverages the knowledge learned from the source task and applies it to the target task. This approach is especially useful when the target task has limited data or computational resources.

In transfer learning, the pre-trained model, also known as the "base model" or "source model," is typically trained on a large dataset and a more general problem (e.g., image classification on ImageNet, a vast dataset with millions of labeled images). The knowledge learned by the base model in the form of feature representations and weights captures common patterns and features in the data.

To perform transfer learning, the following steps are commonly followed:

1. Pre-training: The base model is trained on a source task using a large dataset, which can take a considerable amount of time and computational resources.

2. Feature Extraction: After pre-training, the base model is used as a feature extractor. The last few layers (classifier layers) of the model are discarded, and the remaining layers (feature extraction layers) are retained. These layers serve as feature extractors, producing meaningful representations of the data.

3. Fine-tuning: The feature extraction layers and sometimes some of the earlier layers are connected to a new set of layers, often called the "classifier layers" or "task-specific layers." These layers are randomly initialized, and the model is trained on the target task with a smaller dataset. The weights of the base model can be frozen during fine-tuning, or they can be allowed to be updated with a lower learning rate to fine-tune the model for the target task.

Transfer learning has several benefits:

1. Reduced training time and resource requirements: Since the base model has already learned generic features, transfer learning can save time and resources compared to training a model from scratch.

2. Improved generalization: Transfer learning helps the model generalize better to the target task, especially when the target dataset is small and dissimilar from the source dataset.

3. Better performance: By starting from a model that is already trained on a large dataset, transfer learning can lead to better performance on the target task, especially in scenarios with limited data.

4. Effective feature extraction: The feature extraction layers of the pre-trained model can serve as powerful feature extractors for different tasks, even when the task domains differ.

Transfer learning is commonly used in various domains, including computer vision, natural language processing (NLP), and speech recognition, where pre-trained models are fine-tuned for specific applications like object detection, sentiment analysis, or speech-to-text.

In this mini-project you will perform fine-tuning using Keras with a pre-trained VGG16 model on the CIFAR-10 dataset.

First, import all the libraries you'll need.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

I0000 00:00:1786151231.545644   28793 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786151231.605833   28793 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786151233.031251   28793 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


The CIFAR-10 dataset is a widely used benchmark dataset in the field of computer vision and machine learning. It stands for the "Canadian Institute for Advanced Research 10" dataset. CIFAR-10 was created by researchers at the CIFAR institute and was originally introduced as part of the Neural Information Processing Systems (NIPS) 2009 competition.

The dataset consists of 60,000 color images, each of size 32x32 pixels, belonging to ten different classes. Each class contains 6,000 images. The ten classes in CIFAR-10 are:

1. Airplane
2. Automobile
3. Bird
4. Cat
5. Deer
6. Dog
7. Frog
8. Horse
9. Ship
10. Truck

The images are evenly distributed across the classes, making CIFAR-10 a balanced dataset. The dataset is divided into two sets: a training set and a test set. The training set contains 50,000 images, while the test set contains the remaining 10,000 images.

CIFAR-10 is often used for tasks such as image classification, object recognition, and transfer learning experiments. The relatively small size of the images and the variety of classes make it a challenging dataset for training machine learning models, especially deep neural networks. It also serves as a good dataset for teaching and learning purposes due to its manageable size and straightforward class labels.

Here are your tasks:

1. Load the CIFAR-10 dataset after referencing the documentation [here](https://keras.io/api/datasets/cifar10/).
2. Normalize the pixel values so they're all in the range [0, 1].
3. Apply One Hot Encoding to the train and test labels using the [to_categorical](https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical) function.
4. Further split the the training data into training and validation sets using [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html). Use only 10% of the data for validation.  

In [2]:
# Load the CIFAR-10 dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [3]:
# Normalize the pixel values to [0, 1]

X_train_normalized = X_train / 255.0
X_test_normalized = X_test / 255.0

print("X_train:  %s" % X_train[0][0][0:5])
print("X_train_normalized:  %s" % X_train_normalized[0][0][0:5])

print("y_train:  %s" % y_train)


X_train:  [[59 62 63]
 [43 46 45]
 [50 48 43]
 [68 54 42]
 [98 73 52]]
X_train_normalized:  [[0.23137255 0.24313725 0.24705882]
 [0.16862745 0.18039216 0.17647059]
 [0.19607843 0.18823529 0.16862745]
 [0.26666667 0.21176471 0.16470588]
 [0.38431373 0.28627451 0.20392157]]
y_train:  [[6]
 [9]
 [9]
 ...
 [9]
 [1]
 [1]]


In [4]:
# One-hot encode the labels

y_train_one_hot = to_categorical(y_train)
y_test_one_hot = to_categorical(y_test)

print("y_train_one_hot:  %s" % y_train_one_hot)
print("y_test_one_hot:  %s" % y_test_one_hot)

y_train_one_hot:  [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]]
y_test_one_hot:  [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]]


In [5]:
# Split the data into training and validation sets
X_normalized_training, X_normalized_validation, y_one_hot_training, y_one_hot_validation = \
  train_test_split(X_train_normalized, y_train_one_hot, test_size=0.10, random_state=0)

print("y_one_hot_validation:  %s" % y_one_hot_validation)

y_one_hot_validation:  [[0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]


VGG16 (Visual Geometry Group 16) is a deep convolutional neural network architecture that was developed by the Visual Geometry Group at the University of Oxford. It was proposed by researchers Karen Simonyan and Andrew Zisserman in their paper titled "Very Deep Convolutional Networks for Large-Scale Image Recognition," which was presented at the International Conference on Learning Representations (ICLR) in 2015.

The VGG16 architecture gained significant popularity for its simplicity and effectiveness in image classification tasks. It was one of the pioneering models that demonstrated the power of deeper neural networks for visual recognition tasks.

Key characteristics of the VGG16 architecture:

1. Architecture: VGG16 consists of a total of 16 layers, hence the name "16." These layers are stacked one after another, forming a deep neural network.

2. Convolutional Layers: The main building blocks of VGG16 are the convolutional layers. It primarily uses 3x3 convolutional filters throughout the network, which allows it to capture local features effectively.

3. Max Pooling: After each set of convolutional layers, VGG16 applies max-pooling layers with 2x2 filters and stride 2, which halves the spatial dimensions (width and height) of the feature maps and reduces the number of parameters.

4. Fully Connected Layers: Towards the end of the network, VGG16 has fully connected layers that act as a classifier to make predictions based on the learned features.

5. Activation Function: The network uses the Rectified Linear Unit (ReLU) activation function for all hidden layers, which helps with faster convergence during training.

6. Number of Filters: The number of filters in each convolutional layer is relatively small compared to more recent architectures like ResNet or InceptionNet. However, stacking multiple layers allows VGG16 to learn complex hierarchical features.

7. Output Layer: The output layer consists of 1000 units, corresponding to 1000 ImageNet classes. VGG16 was originally trained on the large-scale ImageNet dataset, which contains millions of images from 1000 different classes.

VGG16 was instrumental in showing that increasing the depth of a neural network can significantly improve its performance on image recognition tasks. However, the main drawback of VGG16 is its high number of parameters, making it computationally expensive and memory-intensive to train. Despite this limitation, VGG16 remains an essential benchmark architecture and has paved the way for even deeper and more efficient models in the field of computer vision, such as ResNet, DenseNet, and EfficientNet.

Here are your tasks:

1. Load [VGG16](https://keras.io/api/applications/vgg/#vgg16-function) as a base model. Make sure to exclude the top layer.
2. Freeze all the layers in the base model. We'll be using these weights as a feature extraction layer to forward to layers that are trainable.

In [6]:
# Load the pre-trained VGG16 model (excluding the top classifier)

inputs = tf.keras.Input(shape=(None, None, 3))

vgg16 = VGG16(include_top=False,
              weights="imagenet",
              input_tensor=inputs,
              input_shape=None,
              pooling=None,
              classes=1000,
              classifier_activation="softmax",
              name="vgg16",
             )

E0000 00:00:1786151238.292176   28793 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
# Freeze the layers in the base model
vgg16.trainable = False

Now, we'll add some trainable layers to the base model.

1. Using the base model, add a [GlobalAveragePooling2D](https://keras.io/api/layers/pooling_layers/global_average_pooling2d/) layer, followed by a [Dense](https://keras.io/api/layers/core_layers/dense/) layer of length 256 with ReLU activation. Finally, add a classification layer with 10 units, corresponding to the 10 CIFAR-10 classes, with softmax activation.
2. Create a Keras [Model](https://keras.io/api/models/model/) that takes in approproate inputs and outputs.

In [8]:
# Add a global average pooling layer
pooling = GlobalAveragePooling2D()(vgg16.output)


In [9]:
# Add a fully connected layer with 256 units and ReLU activation
x = Dense(256, activation="relu")(pooling)


In [10]:
# Add the final classification layer with 10 units (for CIFAR-10 classes) and softmax activation
outputs = Dense(10, activation="softmax")(x)


In [11]:
# Create the fine-tuned model
model = Model(inputs=inputs, outputs=outputs)


With your model complete it's time to train it and assess its performance.

1. Compile your model using an appropriate loss function. Feel free to play around with the optimizer, but a good starting optimizer might be Adam with a learning rate of 0.001.
2. Fit your model on the training data. Use the validation data to print the accuracy for each epoch. Try training for 10 epochs. Note, training can take a few hours so go ahead and grab a cup of coffee.

**Optional**: See if you can implement an [Early Stopping](https://keras.io/api/callbacks/early_stopping/) criteria as a callback function.

In [12]:
# Compile the model
model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

In [13]:
# Train the model
from tensorflow.keras.callbacks import EarlyStopping

monitor_val_acc = EarlyStopping(monitor='accuracy',
                                patience=2)

model.fit(X_normalized_training, y_one_hot_training, epochs=10, callbacks=[ monitor_val_acc ])

Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 89s 63ms/step - accuracy: 0.5264 - loss: 1.3515
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 92s 65ms/step - accuracy: 0.5904 - loss: 1.1737
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 85s 61ms/step - accuracy: 0.6125 - loss: 1.1083
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 89s 63ms/step - accuracy: 0.6275 - loss: 1.0575
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 87s 62ms/step - accuracy: 0.6427 - loss: 1.0161
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 84s 60ms/step - accuracy: 0.6558 - loss: 0.9779
Epoch 7/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 84s 60ms/step - accuracy: 0.6694 - loss: 0.9417
Epoch 8/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 86s 61ms/step - accuracy: 0.6824 - loss: 0.9065
Epoch 9/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 84s 59ms/step - accuracy: 0.6913 - loss: 0.8780
Epoch 10/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 87s 62ms/step - accuracy: 0.7016 - loss: 0.8459


With your model trained, it's time to assess how well it performs on the test data.

1. Use your trained model to calculate the accuracy on the test set. Is the model performance better than random?
2. Experiment! See if you can tweak your model to improve performance.  

In [14]:
# Evaluate the model on the test set

accuracy = model.evaluate(X_normalized_validation, y_one_hot_validation)[1]

print("Accuracy:  %s" % accuracy)

157/157 ━━━━━━━━━━━━━━━━━━━━ 10s 59ms/step - accuracy: 0.6212 - loss: 1.1043
Accuracy:  0.6212000250816345


In [15]:
# The model performance of 62% is better than random (50%) but not great yet.


In [17]:
# Test the VGG16 model itself trained on the input data, for the purpose of comparison

inputs_standalone = tf.keras.Input(shape=(None, None, 3))

vgg16_standalone = VGG16(include_top=False,
                         weights="imagenet",
                         input_tensor=inputs_standalone,
                         input_shape=None,
                         pooling=None,
                         classes=1000,
                         classifier_activation="softmax",
                         name="vgg16_standalone",
                        )

vgg16_standalone.trainable = True

outputs_standalone = Dense(10, activation="softmax")(GlobalAveragePooling2D()(vgg16_standalone.output))

model_standalone = Model(inputs=inputs_standalone, outputs=outputs_standalone)

model_standalone.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

model_standalone.fit(X_normalized_training, y_one_hot_training, epochs=10)


Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1115s 791ms/step - accuracy: 0.3097 - loss: 1.7675
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1111s 789ms/step - accuracy: 0.5783 - loss: 1.1732
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1113s 791ms/step - accuracy: 0.6872 - loss: 0.8983
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1036s 736ms/step - accuracy: 0.7438 - loss: 0.7511
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1021s 726ms/step - accuracy: 0.7810 - loss: 0.6528
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1022s 726ms/step - accuracy: 0.8097 - loss: 0.5672
Epoch 7/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1021s 726ms/step - accuracy: 0.8250 - loss: 0.5273
Epoch 8/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1023s 727ms/step - accuracy: 0.8507 - loss: 0.4489
Epoch 9/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1023s 727ms/step - accuracy: 0.8696 - loss: 0.4003
Epoch 10/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 1027s 730ms/step - accuracy: 0.8797 - loss: 0.3663


In [18]:
accuracy_standalone = model_standalone.evaluate(X_normalized_validation, y_one_hot_validation)[1]

print("VGG16 standalone retrained accuracy:  %s" % accuracy_standalone)

157/157 ━━━━━━━━━━━━━━━━━━━━ 10s 62ms/step - accuracy: 0.7890 - loss: 0.6675
VGG16 standalone retrained accuracy:  0.7889999747276306


In [19]:
# Test the VGG16 model receiving no additional training (on the input data) and no fine-tuning, also for the purpose of comparison

inputs_standalone_frozen = tf.keras.Input(shape=(None, None, 3))

vgg16_standalone_frozen = VGG16(include_top=False,
                         weights="imagenet",
                         input_tensor=inputs_standalone_frozen,
                         input_shape=None,
                         pooling=None,
                         classes=1000,
                         classifier_activation="softmax",
                         name="vgg16_standalone",
                        )

vgg16_standalone_frozen.trainable = False

outputs_standalone_frozen = Dense(10, activation="softmax")(GlobalAveragePooling2D()(vgg16_standalone_frozen.output))

model_standalone_frozen = Model(inputs=inputs_standalone_frozen, outputs=outputs_standalone_frozen)

model_standalone_frozen.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])

model_standalone_frozen.fit(X_normalized_training, y_one_hot_training, epochs=10)


Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 83s 59ms/step - accuracy: 0.4717 - loss: 1.5527
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 83s 59ms/step - accuracy: 0.5486 - loss: 1.3233
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 83s 59ms/step - accuracy: 0.5682 - loss: 1.2622
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 83s 59ms/step - accuracy: 0.5802 - loss: 1.2273
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 89s 63ms/step - accuracy: 0.5856 - loss: 1.2046
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 89s 63ms/step - accuracy: 0.5931 - loss: 1.1880
Epoch 7/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 89s 63ms/step - accuracy: 0.5964 - loss: 1.1753
Epoch 8/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 92s 65ms/step - accuracy: 0.5999 - loss: 1.1653
Epoch 9/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 90s 64ms/step - accuracy: 0.6036 - loss: 1.1568
Epoch 10/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 85s 60ms/step - accuracy: 0.6040 - loss: 1.1505


In [21]:
accuracy_standalone_frozen = model_standalone_frozen.evaluate(X_normalized_validation, y_one_hot_validation)[1]

print("VGG16 standalone un-retrained accuracy and no fine-tuning:  %s" % accuracy_standalone_frozen)

157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 58ms/step - accuracy: 0.5896 - loss: 1.1784
VGG16 standalone un-retrained accuracy and no fine-tuning:  0.5896000266075134


In [ ]:
# The fine-tuned model performance of 62% is better than the non-fined-tuned performance of 59%
# and better than random performance of 50%.
#
# But the VGG baseline model *unfrozen and with its base layers re-trained* (taking more than 10 times longer to train
# than the fine-tuned model with frozen baseline layers) has better performance at 79%.
